# Paso 2 — Flood History + Water-derived Topography
Fuentes combinadas:
- **JRC Global Surface Water** (1984–presente, ~40 años) — historia larga desde Landsat
- **Sentinel-1 SAR** (2014–2024) — detección bajo nubes, eventos recientes

Output: raster de frecuencia de inundación combinado → topografía relativa desde agua.

In [ ]:
import sys
sys.path.insert(0, '..')
from src.config import load_config, get_bbox
from src.gee import init_gee, get_chirps_events, get_jrc_water, export_to_drive
from src.flood import build_flood_inventory, combine_jrc_sar_frequency
from src.water_topo import frequency_to_relative_elevation, compare_dem_vs_water_topo
from src.viz import plot_raster
from pathlib import Path

cfg = load_config()
init_gee()

bbox_ee = get_bbox(cfg, as_ee=True)
processed_dir = Path('../data/processed')
raw_dir = Path('../data/raw')

## 2.1 JRC Global Surface Water (1984–presente)

In [ ]:
jrc = get_jrc_water(bbox_ee)

# Vista rápida de estadísticas JRC en la zona
stats = jrc['occurrence'].reduceRegion(
    reducer=__import__('ee').Reducer.percentile([25, 50, 75, 90, 99]),
    geometry=bbox_ee,
    scale=30,
    maxPixels=1e9,
).getInfo()
print('JRC occurrence percentiles:', stats)

# Exportar las 3 capas JRC a Drive
for layer in ['occurrence', 'seasonality', 'recurrence']:
    export_to_drive(jrc[layer], f'jrc_{layer}', bbox_ee, scale=30)
    print(f'Export iniciado: jrc_{layer}')

## 2.2 Sentinel-1 SAR — eventos de lluvia intensa (2014–2024)

In [ ]:
event_dates = get_chirps_events(
    bbox_ee,
    start=cfg['sar']['start_date'],
    end=cfg['sar']['end_date'],
    threshold_mm=cfg['chirps']['event_threshold_mm'],
)
print(f'{len(event_dates)} eventos de lluvia intensa encontrados')

sar_inventory = build_flood_inventory(bbox_ee, event_dates, polarization=cfg['sar']['polarization'])

# Frecuencia combinada JRC (60%) + SAR (40%)
flood_freq_combined = combine_jrc_sar_frequency(jrc, sar_inventory, bbox_ee)

# Exportar a Drive → descargar a data/raw/flood_freq_combined.tif
export_to_drive(flood_freq_combined, 'flood_freq_combined', bbox_ee, scale=30)
print('Export combinado iniciado')

## 2.3 Topografía relativa desde agua
Después de descargar `flood_freq_combined.tif` de Drive a `data/raw/`:

In [ ]:
rel_elev = frequency_to_relative_elevation(
    flood_freq_path=raw_dir / 'flood_freq_combined.tif',
    output_path=processed_dir / 'water_rel_elev.tif',
)
plot_raster(processed_dir / 'water_rel_elev.tif',
            title='Elevación relativa (desde frecuencia de inundación combinada)',
            cmap='terrain',
            output_path='../outputs/water_rel_elev.png')

## 2.4 Validación: DEM vs topografía desde agua
Spearman r < 0.4 → usar water_rel_elev como feature de elevación en el ML (el DEM tiene demasiado error en terreno plano).

In [ ]:
corr = compare_dem_vs_water_topo(
    dem_path=processed_dir / 'pit_filled.tif',
    rel_elev_path=processed_dir / 'water_rel_elev.tif',
    output_path=processed_dir / 'dem_water_discrepancy.tif',
)
if corr < 0.4:
    print(f'Spearman r={corr:.3f} → usar water_rel_elev como feature de elevación en ML.')
else:
    print(f'Spearman r={corr:.3f} → DEM y water_topo coinciden. Usar ambos como features.')